# 🏠 Project 1: Regression – Predicting House Prices in India
### Dataset: Bengaluru House Price Data
---

## CELL 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

%matplotlib inline
print('✅ All libraries imported successfully!')

---
## STEP 1: Load & Explore Dataset

In [ ]:
# Load dataset
df = pd.read_csv('Bengaluru_House_Data.csv').copy()

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Data types
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())

In [ ]:
# Statistical summary
print('Statistical Summary:')
df.describe()

---
## STEP 2: Data Cleaning & Preprocessing

In [ ]:
# Drop unnecessary columns
cols_to_drop = ['area_type', 'society', 'balcony', 'availability']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

# Drop missing values
df.dropna(inplace=True)

# Fix BHK column
df.loc[:, 'bhk'] = df['size'].apply(lambda x: int(str(x).split(' ')[0]))

# Fix total_sqft - convert ranges to average
def convert_sqft(x):
    try:
        if isinstance(x, str):
            tokens = x.split('-')
            if len(tokens) == 2:
                return (float(tokens[0]) + float(tokens[1])) / 2
        return float(x)
    except:
        return None

df.loc[:, 'total_sqft'] = df['total_sqft'].apply(convert_sqft)
df.dropna(inplace=True)

# Remove outliers (min 300 sqft per room)
df = df[df['total_sqft'] / df['bhk'] >= 300].copy()

print('✅ Cleaned Shape:', df.shape)
print('Columns:', df.columns.tolist())

---
## STEP 3: Feature Engineering

In [ ]:
# Create price per sqft
df.loc[:, 'price_per_sqft'] = df['price'] * 100000 / df['total_sqft']

# Handle location - group rare locations as 'other'
location_counts = df['location'].value_counts()
df.loc[:, 'location'] = df['location'].apply(
    lambda x: x if location_counts[x] > 10 else 'other'
)

# Remove price outliers per location
def remove_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        reduced = subdf[
            (subdf.price_per_sqft > (m - st)) &
            (subdf.price_per_sqft <= (m + st))
        ]
        df_out = pd.concat([df_out, reduced], ignore_index=True)
    return df_out

df = remove_outliers(df)
print('✅ After outlier removal:', df.shape)
df.head()

---
## STEP 4: Exploratory Data Analysis (EDA) – 15 Plots

In [ ]:
# Plot 1: Price Distribution
plt.figure(figsize=(10,5))
sns.histplot(df['price'], bins=50, kde=True, color='blue')
plt.title('Plot 1: Distribution of House Prices')
plt.xlabel('Price (Lakhs)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Plot 2: Total Sqft Distribution
plt.figure(figsize=(10,5))
sns.histplot(df['total_sqft'], bins=50, kde=True, color='green')
plt.title('Plot 2: Distribution of Total Square Footage')
plt.xlabel('Total Sqft')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Plot 3: BHK Distribution
plt.figure(figsize=(8,5))
df['bhk'].value_counts().plot(kind='bar', color='orange', edgecolor='black')
plt.title('Plot 3: BHK Count Distribution')
plt.xlabel('BHK')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Plot 4: Bathroom Distribution
plt.figure(figsize=(8,5))
df['bath'].value_counts().plot(kind='bar', color='purple', edgecolor='black')
plt.title('Plot 4: Bathroom Count Distribution')
plt.xlabel('Bathrooms')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Plot 5: Price vs Total Sqft
plt.figure(figsize=(10,6))
plt.scatter(df['total_sqft'], df['price'], alpha=0.3, color='teal')
plt.title('Plot 5: Price vs Total Sqft')
plt.xlabel('Total Sqft')
plt.ylabel('Price (Lakhs)')
plt.show()

In [ ]:
# Plot 6: Price vs BHK Boxplot
plt.figure(figsize=(10,6))
sns.boxplot(x='bhk', y='price', data=df, palette='Set2')
plt.title('Plot 6: Price vs BHK')
plt.xlabel('BHK')
plt.ylabel('Price (Lakhs)')
plt.show()

In [ ]:
# Plot 7: Price vs Bathrooms
plt.figure(figsize=(10,6))
sns.boxplot(x='bath', y='price', data=df, palette='Set3')
plt.title('Plot 7: Price vs Number of Bathrooms')
plt.xlabel('Bathrooms')
plt.ylabel('Price (Lakhs)')
plt.show()

In [ ]:
# Plot 8: Price per Sqft vs BHK Violin
plt.figure(figsize=(12,6))
sns.violinplot(x='bhk', y='price_per_sqft', data=df, palette='muted')
plt.title('Plot 8: Price per Sqft vs BHK (Violin)')
plt.xlabel('BHK')
plt.ylabel('Price per Sqft')
plt.show()

In [ ]:
# Plot 9: Sqft vs Price colored by BHK
plt.figure(figsize=(10,6))
sns.scatterplot(x='total_sqft', y='price', hue='bhk', data=df, palette='tab10', alpha=0.5)
plt.title('Plot 9: Sqft vs Price by BHK')
plt.xlabel('Total Sqft')
plt.ylabel('Price (Lakhs)')
plt.show()

In [ ]:
# Plot 10: Top 10 Most Expensive Locations
plt.figure(figsize=(12,6))
top_loc = df.groupby('location')['price'].mean().sort_values(ascending=False).head(10)
sns.barplot(x=top_loc.values, y=top_loc.index,
            hue=top_loc.index, palette='Reds_r', legend=False)
plt.title('Plot 10: Top 10 Most Expensive Locations')
plt.xlabel('Average Price (Lakhs)')
plt.show()

In [ ]:
# Plot 11: Top 10 Cheapest Locations
plt.figure(figsize=(12,6))
cheap_loc = df.groupby('location')['price'].mean().sort_values().head(10)
sns.barplot(x=cheap_loc.values, y=cheap_loc.index,
            hue=cheap_loc.index, palette='Greens_r', legend=False)
plt.title('Plot 11: Top 10 Cheapest Locations')
plt.xlabel('Average Price (Lakhs)')
plt.show()

In [ ]:
# Plot 12: Top 15 Locations by Listings
plt.figure(figsize=(12,6))
top15 = df['location'].value_counts().head(15)
sns.barplot(x=top15.values, y=top15.index,
            hue=top15.index, palette='Blues_r', legend=False)
plt.title('Plot 12: Top 15 Locations by Number of Listings')
plt.xlabel('Number of Houses')
plt.show()

In [ ]:
# Plot 13: Correlation Heatmap
plt.figure(figsize=(8,6))
corr = df[['total_sqft', 'bath', 'bhk', 'price', 'price_per_sqft']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Plot 13: Correlation Heatmap')
plt.show()

In [ ]:
# Plot 14: Price per Sqft Distribution
plt.figure(figsize=(10,5))
sns.histplot(df['price_per_sqft'], bins=50, kde=True, color='crimson')
plt.title('Plot 14: Price per Sqft Distribution')
plt.xlabel('Price per Sqft')
plt.show()

In [ ]:
# Plot 15: Pairplot
sns.pairplot(
    df[['total_sqft', 'bath', 'bhk', 'price', 'price_per_sqft']],
    diag_kind='kde',
    plot_kws={'alpha': 0.3}
)
plt.suptitle('Plot 15: Pairplot of All Numeric Features', y=1.02)
plt.show()

---
## STEP 5: Encode Categorical Variables

In [ ]:
# One-Hot Encode location
df_encoded = pd.get_dummies(df, columns=['location'], drop_first=True)

# Drop unused columns
df_encoded.drop(['size', 'price_per_sqft'], axis=1, inplace=True)

print('✅ Encoded Shape:', df_encoded.shape)
df_encoded.head()

---
## STEP 6: Train-Test Split & Scaling

In [ ]:
X = df_encoded.drop('price', axis=1)
y = df_encoded['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('✅ Train size:', X_train.shape)
print('✅ Test size:', X_test.shape)

---
## STEP 7: Model Building

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    results.append({'Model': name, 'RMSE': round(rmse,2),
                    'MAE': round(mae,2), 'R2': round(r2,4)})

    print(f'\n{name}')
    print(f'  RMSE : {rmse:.2f}')
    print(f'  MAE  : {mae:.2f}')
    print(f'  R²   : {r2:.4f}')

---
## STEP 8: Model Evaluation & Comparison

In [ ]:
results_df = pd.DataFrame(results)
print(results_df)

# Plot comparison
plt.figure(figsize=(10,5))
sns.barplot(x='Model', y='R2', data=results_df,
            hue='Model', palette='viridis', legend=False)
plt.title('Model Comparison - R² Score')
plt.ylabel('R² Score')
plt.ylim(0, 1)
for i, row in results_df.iterrows():
    plt.text(i, row['R2'] + 0.01, str(row['R2']), ha='center', fontweight='bold')
plt.show()

In [ ]:
# Best model: Random Forest - Actual vs Predicted
rf_model = models['Random Forest']
y_pred_rf = rf_model.predict(X_test_scaled)

plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred_rf, alpha=0.5, color='darkgreen')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Random Forest: Actual vs Predicted Price')
plt.xlabel('Actual Price (Lakhs)')
plt.ylabel('Predicted Price (Lakhs)')
plt.show()

print('\n✅ Project Complete!')
print('Best Model: Random Forest')
print(f'R² Score: {r2_score(y_test, y_pred_rf):.4f}')